# Data Augmentation with Embedding Model

Text data encoded in embeddings can be used as additional features in tabular dataset as data augmentation.

Sentence transformers are employed to produce embeddings, and then a dimensionality reduction is applied to obtain a desired number of features.

# Prepare Workspace

In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import TruncatedSVD
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import warnings
warnings.simplefilter(action='ignore', category=UserWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
import os
os.environ["HF_TOKEN"] = ""

# Load Model and dataset

In [3]:
# Load a transformer model suitable for sentence embedding, on the GPU
model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2', device='cuda')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
df = pd.read_csv('./work/StormEvents_narratives_filled_fin.csv', low_memory=False)
print(f"Loaded df: {df.shape}")


Loaded df: (1830044, 36)


# Build Embeddings features from episode

In [5]:
# Extract the text data from the last column
text_data_ep = df['EPISODE_NARRATIVE'].tolist()

# Generate embeddings for the text data (larger batch size to keep the GPU busy)
embeddings1 = model.encode(text_data_ep, batch_size=128, show_progress_bar=True)

# Convert embeddings to numpy arrays if needed
embeddings1 = np.array(embeddings1)

# Print the shape of the embeddings to verify the dimension
print(embeddings1.shape)

Batches:   0%|          | 0/14298 [00:00<?, ?it/s]

(1830044, 384)


# Apply Dimensionality Reduction to the Embeddings

In [6]:
# Create a TruncatedSVD instance
n_components = 10  # Desired number of dimensions
svd = TruncatedSVD(n_components=n_components, random_state=0)

# Fit and transform the embeddings
reduced_embeddings1 = svd.fit_transform(embeddings1)

# Check the shape to ensure reduction worked as expected
print(reduced_embeddings1.shape)

(1830044, 10)


# Attach Reduced Embeddings to the Dataset

In [7]:
# make a copy
df1 = df.copy()

# Generate feature names for the reduced dimensions
reduced_feature_names1 = [f"ep_embedding_{i+1}" for i in range(n_components)]

# Add columns to the DataFrame
for i, feature_name in enumerate(reduced_feature_names1):
    df1.loc[:, feature_name] = reduced_embeddings1[:, i]



In [8]:
# Verify the DataFrame to ensure columns are added
df1.head()

,EVENT_ID,STATE,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,...,ep_embedding_1,ep_embedding_2,ep_embedding_3,ep_embedding_4,ep_embedding_5,ep_embedding_6,ep_embedding_7,ep_embedding_8,ep_embedding_9,ep_embedding_10
0,10096222,OKLAHOMA,1950,April,Tornado,C,149,WASHITA,OUN,28-APR-50 14:45:00,...,-0.731984,0.296137,0.001925,-0.201660,0.055463,0.270622,-0.063487,0.184858,0.029797,0.110520
1,10120412,TEXAS,1950,April,Tornado,C,93,COMANCHE,FWD,29-APR-50 15:30:00,...,-0.661453,0.107808,0.072910,-0.366888,-0.034920,0.302603,-0.100861,0.161228,0.240091,-0.036066
2,10104927,PENNSYLVANIA,1950,July,Tornado,C,77,LEHIGH,CTP,05-JUL-50 18:00:00,...,-0.505828,0.101024,0.063997,-0.280795,0.221932,-0.035763,0.051182,0.044068,-0.015401,-0.040811
3,10104928,PENNSYLVANIA,1950,July,Tornado,C,43,DAUPHIN,CTP,05-JUL-50 18:30:00,...,-0.543783,0.047710,0.067663,-0.226650,0.190847,0.029697,0.045080,0.041151,-0.008892,0.010463
4,10104929,PENNSYLVANIA,1950,July,Tornado,C,39,CRAWFORD,CTP,24-JUL-50 14:40:00,...,-0.534143,0.051160,0.064235,-0.224306,0.191206,0.022741,0.040246,0.044590,-0.009231,0.005796


In [11]:
df1.to_csv('./work/StormEvents_episode_embedding.csv', index=False)

# Build Embeddings features from event

In [12]:
# Extract the text data from the last column
text_data_ev = df1['EVENT_NARRATIVE'].tolist()

# Generate embeddings for the text data (larger batch size to keep the GPU busy)
embeddings2 = model.encode(text_data_ev, batch_size=128, show_progress_bar=True)

# Convert embeddings to numpy arrays if needed
embeddings2 = np.array(embeddings2)

# Print the shape of the embeddings to verify the dimension
print(embeddings2.shape)

Batches:   0%|          | 0/14298 [00:00<?, ?it/s]

(1830044, 384)


# Apply Dimensionality Reduction to the Embeddings

In [13]:
# Create a TruncatedSVD instance
n_components = 10  # Desired number of dimensions
svd = TruncatedSVD(n_components=n_components, random_state=0)

# Fit and transform the embeddings
reduced_embeddings2 = svd.fit_transform(embeddings2)

# Check the shape to ensure reduction worked as expected
print(reduced_embeddings2.shape)

(1830044, 10)


# Attach Reduced Embeddings to the Dataset

In [14]:
# make a copy
df2 = df1.copy()

# Generate feature names for the reduced dimensions
reduced_feature_names2 = [f"ev_embedding_{i+1}" for i in range(n_components)]

# Add columns to the DataFrame
for i, feature_name in enumerate(reduced_feature_names2):
    df2.loc[:, feature_name] = reduced_embeddings2[:, i]



In [15]:
# Verify the DataFrame to ensure columns are added
df2.head()

,EVENT_ID,STATE,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,...,ev_embedding_1,ev_embedding_2,ev_embedding_3,ev_embedding_4,ev_embedding_5,ev_embedding_6,ev_embedding_7,ev_embedding_8,ev_embedding_9,ev_embedding_10
0,10096222,OKLAHOMA,1950,April,Tornado,C,149,WASHITA,OUN,28-APR-50 14:45:00,...,-0.496233,-0.193223,-0.057737,0.117990,0.213246,-0.097136,-0.027698,-0.064110,0.144005,0.399115
1,10120412,TEXAS,1950,April,Tornado,C,93,COMANCHE,FWD,29-APR-50 15:30:00,...,-0.581381,-0.243189,-0.043012,0.152483,0.199795,-0.113331,-0.066222,-0.094994,0.078104,0.343238
2,10104927,PENNSYLVANIA,1950,July,Tornado,C,77,LEHIGH,CTP,05-JUL-50 18:00:00,...,-0.511192,-0.240494,-0.083887,0.109519,0.037446,-0.089096,-0.118175,-0.103723,0.178139,0.357282
3,10104928,PENNSYLVANIA,1950,July,Tornado,C,43,DAUPHIN,CTP,05-JUL-50 18:30:00,...,-0.527354,-0.175856,-0.063044,0.137087,0.158303,-0.077884,-0.036516,-0.168490,0.091891,0.352604
4,10104929,PENNSYLVANIA,1950,July,Tornado,C,39,CRAWFORD,CTP,24-JUL-50 14:40:00,...,-0.494164,-0.154628,-0.115622,0.146011,0.243618,-0.074947,-0.016618,-0.128709,0.133232,0.412622


In [16]:
df2.to_csv('./work/StormEvents_fe_embedding.csv', index=False)